In [1]:
import ipympl
%matplotlib ipympl

In [2]:
# setting up logging first or else it gets preempted by another package
import watershed_workflow.ui
watershed_workflow.ui.setup_logging(1)

In [3]:
import os,sys
import logging
import numpy as np
from matplotlib import pyplot as plt
import shapely
import pandas as pd
import geopandas as gpd
pd.options.display.max_columns = None
import pickle

import watershed_workflow 
import watershed_workflow.config
import watershed_workflow.sources
import watershed_workflow.sources.standard_names as names

# set the default figure size for notebooks
plt.rcParams["figure.figsize"] = (8, 6)

## Input: Parameters and other source data

In [4]:
# Force Watershed Workflow to pull data from this directory rather than a shared data directory.
# This picks up the Coweeta-specific datasets set up here to avoid large file downloads for 
# demonstration purposes.
#
def splitPathFull(path):
    """
    Splits an absolute path into a list of components such that
    os.path.join(*splitPathFull(path)) == path
    """
    parts = []
    while True:
        head, tail = os.path.split(path)
        if head == path:  # root on Unix or drive letter with backslash on Windows (e.g., C:\)
            parts.insert(0, head)
            break
        elif tail == path:  # just a single file or directory
            parts.insert(0, tail)
            break
        else:
            parts.insert(0, tail)
            path = head
    return parts

cwd = splitPathFull(os.getcwd())
assert cwd[-1] == 'Neches_new'
cwd = cwd[:-1]

# Note, this directory is where downloaded data will be put as well
data_dir = os.path.join(*(cwd + ['input_data',]))
def toInput(filename):
    return os.path.join(data_dir, filename)

output_dir = os.path.join(*(cwd + ['output_data',]))
output_filenames = dict()
def fromOutput(filename):
    return os.path.join(output_dir, filename)    

def toOutput(role, filename):
    output_filenames[role] = filename
    return fromOutput(filename)

# check output and input dirs exist
if not os.path.isdir(data_dir):
    os.makedirs(data_dir, exist_ok=True)
if not os.path.isdir(output_dir):
    os.makedirs(output_dir, exist_ok=True)

In [5]:
# Set the data directory to the local space to get the locally downloaded files
# REMOVE THIS CELL for general use outside fo Coweeta
watershed_workflow.config.setDataDirectory(data_dir)

In [6]:
## Parameters cell -- this provides all parameters that can be changed via pipelining to generate a new watershed. 
name = 'Neches_River'
hucs = ['1202'] # a list of HUCs to run
neches_shapefile = '/Users/s2t/research/neches/scripts/neches_boundary.shp'
#neches_shapefile = '/Users/s2t/research/neches/scripts/subbasins.shp'
#near_coastal = ('/Users/s2t/research/neches/data/near_coast_huc12s_nhdv2.shp')#

# Geometric parameters
# -- parameters to clean and reduce the river network prior to meshing
prune_by_area = 100              # km^2
simplify = 125                   # length scale to target average edge 

# -- mesh triangle refinement control
refine_d0 = 200
refine_d1 = 600

refine_L0 = 125
refine_L1 = 300

refine_A0 = refine_L0**2 / 2
refine_A1 = refine_L1**2 / 2

min_angle = 20 # degrees

# Note that, by default, we tend to work in the DayMet CRS because this allows us to avoid
# reprojecting meteorological forcing datasets.
crs = watershed_workflow.crs.default_crs

# Reload data


In [12]:
with open(fromOutput('02_watersheds_neches.pickle'), 'rb') as fid:
    watersheds = pickle.load(fid)

reaches = gpd.read_parquet(fromOutput('02_rivers_neches.parquet'))
rivers = watershed_workflow.river_tree.createRivers(reaches, method='native')
#rivers = watershed_workflow.river_tree.createRivers(reaches, method='hydroseq')

TypeError: unhashable type: 'Series'

In [11]:
reaches

,geometry,comid,fdate,resolution,gnis_id,reachcode,flowdir,wbareacomi,ftype,fcode,shape_length,streamleve,streamcalc,fromnode,tonode,levelpathi,pathlength,terminalpa,arbolatesu,startflag,terminalfl,dnlevel,uplevelpat,dnlevelpat,dnminorhyd,dndraincou,frommeas,tomeas,rtndiv,vpuin,vpuout,divdasqkm,tidal,totma,wbareatype,pathtimema,hwnodesqkm,maxelevraw,minelevraw,maxelevsmo,minelevsmo,slope,elevfixed,hwtype,slopelenkm,qa_ma,va_ma,qc_ma,vc_ma,qe_ma,ve_ma,qa_01,va_01,qc_01,vc_01,qe_01,ve_01,qa_02,va_02,qc_02,vc_02,qe_02,ve_02,qa_03,va_03,qc_03,vc_03,qe_03,ve_03,qa_04,va_04,qc_04,vc_04,qe_04,ve_04,qa_05,va_05,qc_05,vc_05,qe_05,ve_05,qa_06,va_06,qc_06,vc_06,qe_06,ve_06,qa_07,va_07,qc_07,vc_07,qe_07,ve_07,qa_08,va_08,qc_08,vc_08,qe_08,ve_08,qa_09,va_09,qc_09,vc_09,qe_09,ve_09,qa_10,va_10,qc_10,vc_10,qe_10,ve_10,qa_11,va_11,qc_11,vc_11,qe_11,ve_11,qa_12,va_12,qc_12,vc_12,qe_12,ve_12,lakefract,surfarea,rareahload,rpuid,vpuid,enabled,gridcode,featureid,sourcefc,shape_length_ca,shape_area,catchment_area,ID,name,length,stream_order,drainage_area_sqkm,catchment,hydroseq,uphydroseq,dnhydroseq,divergence,do-not-merge,parent_ID,children_IDs
new_preorder_index,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,"LINESTRING (204160.504 769618.514, 204355.314 ...",1115825,2008-06-13T04:00:00Z,Medium,1375103,12020003000662,With Digitized,120049974,ArtificialPath,55800,0.033209,1,6,630001649.0,630001652.0,630002611.0,48.456,630002611.0,14909.296,0,0,1,630002611.0,630002611.0,0.0,1,0.08398,100.00000,1,0,0,26058.0528,1,-9999.0,StreamRiver,-9999.0,NaN,-9998.0,0.0,0.0,0.0,0.000010,0,,3.333,10502.397,-9999.0,6136.253,-9999.0,8155.942,-9999.0,23802.300,-9999.0,19664.256,-9999.0,11590.213,-9999.0,22459.401,-9999.0,17617.395,-9999.0,12421.598,-9999.0,19825.867,-9999.0,12394.354,-9999.0,13279.914,-9999.0,13540.472,-9999.0,13205.796,-9999.0,12211.358,-9999.0,9481.628,-9999.0,6227.619,-9999.0,10265.883,-9999.0,5897.946,-9999.0,7132.510,-9999.0,9611.261,-9999.0,3276.315,-9999.0,3721.922,-9999.0,7005.287,-9999.0,2320.208,-9999.0,2477.157,-9999.0,4387.282,-9999.0,2294.354,-9999.0,3372.287,-9999.0,4276.148,-9999.0,2226.347,-9998,3853.056,-9998,4622.457,-9998,4483.332,-9999.0,14952.909,-9999.0,10452.280,-9999.0,16497.243,-9999.0,13035.140,-9999.0,7896.345,-9999.0,NaN,NaN,NaN,12a,12,1,1606746.0,1115825.0,NHDFlowline,0.125780,0.000510,12.1644,1115825,Neches River,3.333,6,26058.0528,"POLYGON ((201881.698 767598.233, 201956.95 767...",630002782.0,630002787.0,630002777.0,2,-1,<NA>,[1]
1,"LINESTRING (203236.553 769856.583, 203407.859 ...",1115819,2008-06-13T04:00:00Z,Medium,1375103,12020003000682,With Digitized,120049974,ArtificialPath,55800,0.009913,1,6,630001647.0,630001649.0,630002611.0,51.789,630002611.0,14902.640,0,0,1,630002611.0,630002611.0,0.0,1,0.00000,100.00000,1,0,0,26045.8884,1,-9999.0,StreamRiver,-9999.0,NaN,-9998.0,0.0,0.0,0.0,0.000010,0,,0.967,10496.641,-9999.0,6133.507,-9999.0,8153.196,-9999.0,23787.650,-9999.0,19653.526,-9999.0,11579.482,-9999.0,22448.153,-9999.0,17609.775,-9999.0,12413.978,-9999.0,19817.201,-9999.0,12389.763,-9999.0,13275.323,-9999.0,13535.799,-9999.0,13201.697,-9999.0,12207.259,-9999.0,9477.603,-9999.0,6225.452,-9999.0,10263.715,-9999.0,5895.158,-9999.0,7129.613,-9999.0,9608.364,-9999.0,3274.483,-9999.0,3719.967,-9999.0,7003.332,-9999.0,2318.919,-9999.0,2475.840,-9999.0,4385.965,-9999.0,2292.974,-9999.0,3370.149,-9999.0,4274.010,-9999.0,2225.153,-9998,3851.035,-9998,4620.437,-9998,4476.582,-9999.0,14927.748,-9999.0,10427.119,-9999.0,16486.446,-9999.0,13027.870,-9999.0,7889.076,-9999.0,NaN,NaN,NaN,12a,12,1,1606420.0,1115819.0,NHDFlowline,0.054832,0.000116,3.5028,1115819,Neches River,0.967,6,26045.8884,"POLYGON ((202622.103 768846.58, 202728.295 768...",630002787.0,630002792.0,630002782.0,2,0,1115825,[2]
2,"LINESTRING (202662.473 769898.582, 202854.399 ...",1115813,2008-06-13T04:00:00Z,Medium,1375103,12020003000676,With Digitized,120049974,